# Data Owner — Auto-Approve Demo

This notebook sets up a **Data Owner** on SyftBox with:
- Background services for automatic job approval and email notifications
- An approved script that a Data Scientist can submit

**Run this notebook first**, then have the DS run their notebook.

## 1. Install

In [ ]:
!pip install -q syft-client syft-bg

## 2. Configuration

Set your emails below.

In [ ]:
DO_EMAIL = "your-do-email@example.com"   # Your email (Data Owner)
DS_EMAIL = "your-ds-email@example.com"   # Data Scientist's email

## 3. Initialize syft-bg

Creates config and starts background services (notify + approve).

In [ ]:
import syft_bg

result = syft_bg.init(email=DO_EMAIL, start=True)
result

## 4. Authenticate (first time only)

Sets up Gmail and Drive tokens. Skip this cell if you've already authenticated.

In [ ]:
# Uncomment and run if init() reports missing tokens
# syft_bg.authenticate()

## 5. Login as Data Owner

In [ ]:
from syft_client import login_do

do_client = login_do(email=DO_EMAIL)

## 6. Check peers

In [ ]:
do_client.peers

## 7. Create the approved script

This is the script the DS will submit. The DO registers it for auto-approval.

In [ ]:
%%writefile main.py
import syft_client as sc

data_path = "syft://private/syft_datasets/Sales Dataset/sales.csv"
resolved_path = sc.resolve_path(data_path)

import pandas as pd

df = pd.read_csv(resolved_path)
total = (df["quantity"] * df["price_per_unit"]).sum()
print(f"Total Sales: {total}")

## 8. Register auto-approval

Jobs from `DS_EMAIL` that contain this exact script will be approved automatically.

In [ ]:
result = syft_bg.auto_approve(
    contents=["main.py"],
    peers=[DS_EMAIL],
)
result

## 9. Check status

Verify services are running and the approval object is configured.

In [ ]:
syft_bg.status

## 10. Wait for DS to submit a job

Have the DS run their notebook now. Once they submit, the job will appear here.

In [ ]:
do_client.jobs

## 11. Review results

Once the job is auto-approved and executed, view the output.

In [ ]:
do_client.jobs

In [ ]:
do_client.jobs[0].stdout

## 12. Check service logs

In [ ]:
print("--- approve logs ---")
for line in syft_bg.logs("approve", n=20):
    print(line)

print("\n--- notify logs ---")
for line in syft_bg.logs("notify", n=20):
    print(line)

## 13. Cleanup

In [ ]:
syft_bg.stop()